# Классификация видов пингвинов с помощью KNN

Задача проекта: определить вид пингвина по его физическим характеристикам и посмотреть, как число соседей влияет на качество и форму решающей границы.

В работе используется готовый `KNeighborsClassifier`, а затем тот же алгоритм реализуется с нуля на NumPy.

## Загрузка и подготовка данных

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv('penguins_data.csv')
data.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'penguins_data.csv'

In [ ]:
data.shape

In [ ]:
data.isna().sum()

In [ ]:
data = data.dropna()
data.isna().sum()

In [ ]:
print(data['Island'].nunique())
print(data['Species'].nunique())


In [ ]:
data = pd.get_dummies(data, columns=['Island', 'Sex'], drop_first=True)

data.head()

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

species_series = data['Species']

encoder = OrdinalEncoder(categories=[['Chinstrap', 'Gentoo', 'Adelie']])
data['Species'] = encoder.fit_transform(species_series.to_frame(name='Species')).astype(int)

data.head()

In [ ]:
X = data.drop('Species', axis=1)
y = data['Species']

print(X.head())
print()
print(y.head())

Категориальные признаки кодируются, целевая переменная переводится в числовой формат. После очистки данные разделяются на train и test.

## Train/test split и масштабирование

In [ ]:
from sklearn.model_selection import train_test_split

np.random.seed(67)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=67, test_size=0.3, stratify=y)

print(X_train.shape)
print(X_test.shape)
print()
print(y_train.shape)
print(y_test.shape)
print()
print(X_train.shape[0] / data.shape[0]*100)
print(X_test.shape[0] / data.shape[0]*100)


In [ ]:
X_train_new = X_train[['Flipper Length (mm)', 'Body Mass (g)']]
X_test_new = X_test[['Flipper Length (mm)', 'Body Mass (g)']]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_new)
X_test_scaled = scaler.transform(X_test_new)

`StandardScaler` обучается только на train. Для визуализации используются длина ласта и масса тела.

## KNN из scikit-learn

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
models_with_diff_k = []

for i in [1, 3, 5, 10, 15, 25]:
    knn = KNeighborsClassifier(n_neighbors=i, metric='euclidean')
    knn.fit(X_train_scaled, y_train)
    models_with_diff_k.append(knn)


    print(f'k = {i}')
    print(f'Train accuracy = {knn.score(X_train_scaled, y_train)}')
    print(f'Test accuracy = {knn.score(X_test_scaled, y_test)}')
    print()

Проверяются несколько значений `k`. Маленькое `k` дает более сложную и чувствительную к отдельным объектам границу. При увеличении `k` решение становится более гладким.

## Визуализация решающих границ

In [ ]:
import matplotlib.pyplot as plt
from mlxtend.plotting import plot_decision_regions
import matplotlib.gridspec as gridspec
import itertools
gs = gridspec.GridSpec(2, 3)

fig = plt.figure(figsize=(16, 9))

labels = ['k=1', 'k=3', 'k=5', 'k= 10', 'k=15', 'k= 25']
X_for_vis = X_train_scaled
y_for_vis = y_train.to_numpy()
for clf, lab, grd in zip(models_with_diff_k, labels, itertools.product([0, 1], [0, 1, 2])):

    ax = plt.subplot(gs[grd[0], grd[1]])
    fig = plot_decision_regions(X=X_for_vis, y=y_for_vis, clf=clf, legend=2)
    plt.title(lab)

plt.show()

## KNN с нуля

In [ ]:
import numpy as np
from collections import Counter

class KNN:
    def __init__(self, k:int):
        self.k = k

    def fit(self, X, y):
        self.X_train= np.array(X)
        self.y_train = np.array(y)

    def predict(self, X):
        X = np.array(X)
        ans = []
        for x in X:
            dist = self.count_distance(x,self.X_train)
            closest =np.argsort(dist)[:self.k]
            closest_classes=self.y_train[closest]
            ans.append(Counter(closest_classes).most_common(1)[0][0])

        return np.array(ans)

    def count_distance(self, x, y):
        return np.linalg.norm(y-x, axis=1)

Собственная реализация сохраняет train, считает евклидовы расстояния до нового объекта, выбирает ближайших соседей и определяет класс голосованием.

## Проверка собственной реализации

In [ ]:
best_k_for_penguin = -67456748
best_acc_for_penguin = -562462

for k in range(30,41):
    knn = KNN(k= k)
    knn.fit(X_train_scaled, y_train)
    y_pred_test_penguin= knn.predict(X_test_scaled)
    test_acc = (y_pred_test_penguin==y_test).mean()
    if test_acc > best_acc_for_penguin:
        best_acc_for_penguin = test_acc
        best_k_for_penguin = k

print(f'Best k ={best_k_for_penguin}')
print(f'Best accuracy = {best_acc_for_penguin}')

В дополнительном переборе значений `k` лучший результат собственной реализации получился при `k=40`, accuracy около `0.786`.